# 04b · Segment Anything 2.1 — your four stamped-part photos
Standalone alternative to YOLOE: no training, no class vocabulary or prior model bundle.
No photos are bundled. Set IMAGE_PATH to a Drive image/folder or leave it blank
to upload your own photos. Dependencies and weights need internet on first use.
Choose **Runtime → Change runtime type → T4 GPU** before running. CPU works but is slow.

Default: SAM 2.1 Large, automatic grid-of-points mask generation. This proposes regions,
not guaranteed one-mask-per-part instances. It may split a part, include shadows/holes,
return overlapping/nested masks or miss parts. It does NOT identify types or defects.
An optional box-prompt cell lets you request a particular part without training.
No ground-truth masks exist here: predicted IoU is a model score, NOT measured accuracy.

User-supplied images: no open redistribution license established; verify rights before
publishing this notebook. Model/code: https://github.com/facebookresearch/sam2
Apache 2.0; review upstream licenses. No notebook/model execution by the coding agent.


In [ ]:
# Change these values BEFORE running subsequent cells.
USE_DRIVE = True
ROOT = '/content/drive/MyDrive/Robotekin2026'
SEED = 42
DATASET_PATH = ''  # Existing dataset folder on Drive; empty opens upload.
MODEL_PATH = ''    # Existing exported model folder; empty opens upload.
RESUME_RUN = ''    # Existing run folder for checkpoint continuation.
MODEL_ID = 'facebook/sam2.1-hiera-large'  # Also: facebook/sam2.1-hiera-small
MODEL_REVISION = 'main'  # Resolved checkpoint revision/hash is recorded; pin it for repeat runs
SAM2_COMMIT = '2b90b9f5ceec907a1c18123530e92e794ad901a4'
IMAGE_PATH = ''  # Drive image/folder; blank uploads one or more photos
POINTS_PER_SIDE = 32
POINTS_PER_BATCH = 16  # Lower to 8 or 4 if GPU memory is insufficient
CROP_N_LAYERS = 0  # Try 1 for small/missed parts; more compute
PRED_IOU_THRESH = 0.8
STABILITY_SCORE_THRESH = 0.95
USE_M2M = True
MIN_MASK_AREA = 100
MAX_MASK_FRACTION = 0.85  # Optional large-background filter, not ground truth
MAX_MASK_PREVIEWS = 16
BOX_IMAGE_INDEX = 0
BOX_XYXY = []  # Optional [x1,y1,x2,y2] in original image pixels
CPU_THREADS = 2


In [ ]:
import os, random, subprocess, sys, uuid
from datetime import datetime, timezone
from pathlib import Path
print('Python:', sys.version, '| executable:', sys.executable, flush=True)
install_log = Path('/content/roboqc_pip_install.log')
install_log.parent.mkdir(parents=True, exist_ok=True)
command = [sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', 'numpy>=1.26,<3', 'torch>=2.5.1,<3', 'torchvision>=0.20.1,<1', 'matplotlib>=3.9,<4', 'Pillow>=10,<13', 'opencv-python-headless>=4.10,<6', 'huggingface-hub>=0.26,<2', 'hydra-core>=1.3.2,<2', 'iopath>=0.1.10,<1', 'tqdm>=4.66,<5']
with install_log.open('w', encoding='utf-8') as log:
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
        log.write(line)
    exit_code = process.wait()
if exit_code:
    raise RuntimeError(f'pip failed (exit {exit_code}). Read the ERROR lines above. Full log: {install_log}')
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
else:
    ROOT = '/content/Robotekin2026'
RUN = Path(RESUME_RUN) if RESUME_RUN else Path(ROOT) / (datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:6])
RUN.mkdir(parents=True, exist_ok=bool(RESUME_RUN))
import numpy as np, torch
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
GPU = torch.cuda.is_available()
DEVICE = 'cuda' if GPU else 'cpu'
print('Device:', DEVICE, '| Run:', RUN)
(RUN / 'environment.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))


In [ ]:
# Disable optional CUDA extension compilation; preserve mask holes (no fill-hole postprocess).
os.environ['SAM2_BUILD_CUDA'] = '0'
command = [sys.executable,'-m','pip','install','--no-build-isolation',
           'git+https://github.com/facebookresearch/sam2.git@' + SAM2_COMMIT]
with install_log.open('a',encoding='utf-8') as log:
    process = subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in process.stdout:
        print(line,end='',flush=True); log.write(line)
    if process.wait(): raise RuntimeError(f'SAM 2 install failed. Full log: {install_log}')
(RUN / 'environment.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
torch.set_num_threads(CPU_THREADS)
if not GPU:
    print('WARNING: CPU runtime. Large automatic segmentation can be very slow. GPU recommended.')


In [ ]:
"""Photo input helper embedded as source in public notebooks; never embeds photos."""

from pathlib import Path
import uuid


def select_image_paths(image_path, run, uploader=None):
    suffixes = {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tif', '.tiff'}
    if image_path:
        source = Path(image_path).resolve()
        if not source.exists():
            raise FileNotFoundError(source)
    else:
        if uploader is None:
            from google.colab import files
            uploader = files.upload
        uploaded = uploader()
        if not uploaded:
            raise ValueError('Upload one or more photos, or set IMAGE_PATH to a Drive image/folder')
        if any(Path(name).name != name or '\\' in name or ':' in name or
               Path(name).suffix.lower() not in suffixes for name in uploaded):
            raise ValueError('Upload image files only (not ZIPs); use IMAGE_PATH for nested folders')
        source = Path(run) / ('uploaded_photos_' + uuid.uuid4().hex)
        source.mkdir(parents=True, exist_ok=False)
        for name, blob in uploaded.items():
            with (source / name).open('xb') as handle:
                handle.write(blob)
    paths = [source] if source.is_file() else sorted(p for p in source.rglob('*') if p.is_file())
    paths = [p for p in paths if p.suffix.lower() in suffixes]
    if not paths:
        raise ValueError('No supported photos found; set IMAGE_PATH or upload photos')
    return paths


In [ ]:
import json, shutil, time
from PIL import Image, ImageOps
import matplotlib.pyplot as plt
SOURCES = {'roboqc/__init__.py': '', 'roboqc/industrial/__init__.py': '', 'roboqc/industrial/vision_inference.py': '"""CPU segmentation, planar geometry and optional visible-quality classifier.\n\nThis module intentionally imports no GUI or robot library and is embedded in Colab.\n"""\n\nimport json\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\n\nCLASSES = ["bolt", "nut", "washer", "bracket"]\nQUALITY = ["good", "defect_1", "defect_2"]\n\n\ndef mask_geometry(mask, symmetric=False, clearance_px=3):\n    mask = np.asarray(mask, dtype=np.uint8)\n    ys, xs = np.nonzero(mask)\n    if len(xs) < 3:\n        return {"center_px": None, "orientation_deg": None, "grasp_candidates": []}\n    center = [float(xs.mean()), float(ys.mean())]\n    eigenvalues, eigenvectors = np.linalg.eigh(np.cov(np.column_stack([xs, ys]), rowvar=False))\n    orientation = None\n    if not symmetric and eigenvalues[-1] > 1.3 * max(eigenvalues[0], 1e-9):\n        axis = eigenvectors[:, -1]\n        orientation = float(np.degrees(np.arctan2(axis[1], axis[0])) % 180)\n    distance = cv2.distanceTransform(mask, cv2.DIST_L2, 5)\n    _, radius, _, point = cv2.minMaxLoc(distance)\n    candidates = []\n    if radius >= clearance_px:\n        candidates.append({"position_px": list(point), "clearance_px": float(radius),\n                           "orientation_deg": orientation, "validated": False})\n    return {"center_px": center, "orientation_deg": orientation, "grasp_candidates": candidates}\n\n\ndef classify_mask_color(frame, mask, minimum_fraction=.6):\n    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)\n    # Erode edges to reduce background contamination; classify only saturated pixels.\n    interior = cv2.erode(mask.astype(np.uint8), np.ones((3, 3), np.uint8)) > 0\n    values = hsv[interior]\n    if not len(values):\n        return None, 0.0\n    ranges = {"red": ((values[:, 0] <= 8) | (values[:, 0] >= 170)),\n              "green": (values[:, 0] >= 35) & (values[:, 0] <= 85),\n              "blue": (values[:, 0] >= 86) & (values[:, 0] <= 130)}\n    votes = {key: float(np.mean(region & (values[:, 1] >= 70) & (values[:, 2] >= 45))) for key, region in ranges.items()}\n    color = max(votes, key=votes.get)\n    confidence = votes[color]\n    return (color if confidence >= minimum_fraction else None), confidence\n\n\ndef quality_crop(frame, mask):\n    ys, xs = np.nonzero(mask)\n    if not len(xs):\n        raise ValueError("Empty instance mask")\n    patch = np.where(mask[..., None] > 0, frame, 127)[ys.min():ys.max()+1, xs.min():xs.max()+1]\n    return cv2.cvtColor(cv2.resize(patch, (224, 224)), cv2.COLOR_BGR2RGB)\n\n\nclass Segmenter:\n    def __init__(self, bundle, prefer_onnx=True, confidence=.35):\n        from ultralytics import YOLO\n        self.root = Path(bundle)\n        self.metadata = json.loads((self.root / "vision_manifest.json").read_text(encoding="utf-8"))\n        if self.metadata.get("schema_version") != "industrial-vision-1.0":\n            raise ValueError("Unsupported vision artifact schema")\n        self.classes = self.metadata["classes"]\n        filename = "segmentation.onnx" if prefer_onnx and (self.root / "segmentation.onnx").exists() else "segmentation.pt"\n        if filename.endswith(".pt") and self.metadata.get("model_family") == "yoloe":\n            from ultralytics import YOLOE\n            self.model = YOLOE(str(self.root / filename), task="segment")\n        else:\n            self.model = YOLO(str(self.root / filename), task="segment")\n        self.confidence = confidence\n        self.quality_session = None\n        if self.metadata.get("quality_enabled"):\n            import onnxruntime as ort\n            self.quality_session = ort.InferenceSession(str(self.root / "quality.onnx"), providers=["CPUExecutionProvider"])\n\n    def predict(self, frame):\n        # PT otherwise uses minimal rectangular padding, unlike static square ONNX.\n        # YOLOE also defaults to agnostic NMS while YOLO(onnx) does not.\n        prediction = self.model.predict(frame, device="cpu", imgsz=self.metadata.get("imgsz", 640),\n            conf=self.confidence, rect=False, agnostic_nms=self.metadata.get("model_family") == "yoloe",\n            iou=.7, max_det=300, retina_masks=False, augment=False, verbose=False)[0]\n        objects = []\n        if prediction.masks is None:\n            return objects\n        masks = prediction.masks.data.cpu().numpy()\n        height, width = frame.shape[:2]\n        for i, (box, cls, confidence) in enumerate(zip(prediction.boxes.xyxy.cpu().numpy(), prediction.boxes.cls.cpu().numpy(), prediction.boxes.conf.cpu().numpy())):\n            # Ultralytics masks may be in letterboxed coordinates, not frame coordinates.\n            from ultralytics.utils.ops import scale_masks\n            import torch\n            restored = scale_masks(torch.as_tensor(masks[i:i+1, None],dtype=torch.float32), (height, width))[0, 0].numpy()\n            mask = (restored > .5).astype(np.uint8)\n            if not mask.any():\n                continue\n            x1, y1, x2, y2 = box.tolist()\n            if int(cls) != cls or not 0 <= int(cls) < len(self.classes):\n                raise ValueError("Prediction class ID does not match the exported class mapping")\n            object_type = self.classes[int(cls)]\n            color, color_confidence = classify_mask_color(frame, mask)\n            geometry = mask_geometry(mask, symmetric=object_type in ("washer", "nut"))\n            if geometry["center_px"] is None:\n                continue\n            quality, quality_confidence = "unknown", 0.0\n            if self.quality_session is not None:\n                crop = quality_crop(frame, mask).astype(np.float32) / 255\n                x = ((crop - [.485, .456, .406]) / [.229, .224, .225]).transpose(2, 0, 1)[None].astype(np.float32)\n                logits = self.quality_session.run(None, {self.quality_session.get_inputs()[0].name: x})[0][0]\n                p = np.exp(logits - logits.max()); p /= p.sum()\n                quality_confidence = float(p.max())\n                if quality_confidence >= self.metadata.get("quality_threshold", .8):\n                    quality = QUALITY[int(p.argmax())]\n            partial = x1 <= 1 or y1 <= 1 or x2 >= width-1 or y2 >= height-1\n            if partial:\n                quality = "unknown"\n                geometry["grasp_candidates"] = []\n            objects.append({"type": object_type, "confidence": float(confidence), "mask": mask,\n                "bbox": [int(x1), int(y1), int(x2-x1), int(y2-y1)], "color": color,\n                "color_confidence": color_confidence, "quality": quality,\n                "quality_confidence": quality_confidence, "partial": partial, **geometry})\n        return objects\n\n\ndef draw_instances(frame, objects):\n    out = frame.copy()\n    for i, obj in enumerate(objects):\n        mask = obj["mask"] > 0\n        color = np.array([(53*i+60) % 255, (97*i+100) % 255, (131*i+180) % 255])\n        out[mask] = (.55*out[mask] + .45*color).astype(np.uint8)\n        x, y, w, h = obj["bbox"]\n        cv2.rectangle(out, (x, y), (x+w, y+h), color.tolist(), 2)\n        cv2.putText(out, f"{obj[\'type\']} {obj[\'confidence\']:.2f} {obj[\'quality\']}", (x, max(16,y-5)), cv2.FONT_HERSHEY_SIMPLEX, .45, (255,255,255), 1)\n        for point in obj["grasp_candidates"]:\n            cv2.circle(out, tuple(map(int, point["position_px"])), 4, (0,255,255), -1)\n    return out\n', 'roboqc/industrial/sam_masks.py': '"""SAM candidate export utilities. No model loading, downloads or inference."""\n\nimport json\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\n\nfrom .vision_inference import mask_geometry\n\n\ndef save_candidates(rgb, annotations, output, minimum_area=100, maximum_fraction=.85, seed=42):\n    """Preserve all raw masks; export filtered candidates without claiming object labels."""\n    rgb = np.asarray(rgb)\n    if rgb.ndim != 3 or rgb.shape[2] != 3 or rgb.dtype != np.uint8:\n        raise ValueError("Expected a uint8 RGB image")\n    if minimum_area < 1 or not 0 < maximum_fraction <= 1:\n        raise ValueError("Require minimum_area >= 1 and 0 < maximum_fraction <= 1")\n    height,width = rgb.shape[:2]\n    masks = [np.asarray(a["segmentation"],dtype=bool) for a in annotations]\n    if any(m.shape != (height,width) for m in masks):\n        raise ValueError("Every mask must match the original image shape")\n    out = Path(output)\n    out.mkdir(parents=True,exist_ok=False)\n    np.savez_compressed(out / "raw_masks.npz",masks=np.stack(masks) if masks else np.zeros((0,height,width),bool))\n    rows = []\n    for index,(annotation,mask) in enumerate(zip(annotations,masks)):\n        area = int(mask.sum())\n        reason = "below_minimum_area" if area < minimum_area else "above_maximum_fraction" if area > maximum_fraction*height*width else None\n        ys,xs = np.nonzero(mask)\n        box = [int(xs.min()),int(ys.min()),int(xs.max()-xs.min()+1),int(ys.max()-ys.min()+1)] if area else None\n        filename = f"mask_{index:04d}.png" if reason is None else None\n        if filename:\n            if not cv2.imwrite(str(out / filename),mask.astype(np.uint8)*255):\n                raise OSError(f"Could not save mask: {out / filename}")\n        rows.append({"id":index,"raw_mask_index":index,"area_px":area,"bbox_xywh":box,\n            "model_predicted_iou":float(annotation["predicted_iou"]) if annotation.get("predicted_iou") is not None else None,\n            "stability_score":float(annotation["stability_score"]) if annotation.get("stability_score") is not None else None,\n            "detector_score":float(annotation["detector_score"]) if annotation.get("detector_score") is not None else None,\n            "prompt_label":annotation.get("prompt_label"),\n            "kept":reason is None,"filter_reason":reason,"mask_file":filename,\n            "category":"unknown","quality":"unknown",**mask_geometry(mask)})\n    colors = np.random.default_rng(seed).integers(40,240,size=(len(rows),3))\n    overlay = rgb.copy()\n    # Smaller nested regions are drawn last, but all original overlapping masks remain separate.\n    for row in sorted((r for r in rows if r["kept"]),key=lambda r:r["area_px"],reverse=True):\n        mask = masks[row["id"]]\n        overlay[mask] = (.55*overlay[mask] + .45*colors[row["id"]]).astype(np.uint8)\n    for row in rows:\n        if row["kept"]:\n            x,y,w,h = row["bbox_xywh"]\n            cv2.rectangle(overlay,(x,y),(x+w-1,y+h-1),(0,170,0),1)\n            cv2.putText(overlay,str(row["id"]),(x,max(14,y)),cv2.FONT_HERSHEY_SIMPLEX,.5,(180,0,0),1)\n    if not cv2.imwrite(str(out / "overlay.png"),cv2.cvtColor(overlay,cv2.COLOR_RGB2BGR)):\n        raise OSError("Could not save overlay")\n    result = {"schema_version":"sam-candidates-1.0","image_size":[height,width],\n        "raw_masks":len(rows),"kept_candidates":sum(r["kept"] for r in rows),"candidates":rows,\n        "filters":{"minimum_area":minimum_area,"maximum_fraction":maximum_fraction},\n        "accuracy":"not_measured","warning":"Candidates may overlap or describe subparts/background, not unique physical objects"}\n    (out / "candidates.json").write_text(json.dumps(result,indent=2),encoding="utf-8")\n    return result,overlay\n'}
RUNTIME = Path('/content/roboqc_sam_runtime')
for relative,source in SOURCES.items():
    destination = RUNTIME / relative
    destination.parent.mkdir(parents=True,exist_ok=True)
    destination.write_text(source,encoding='utf-8')
sys.path.insert(0,str(RUNTIME))
from roboqc.industrial.sam_masks import save_candidates
paths = select_image_paths(IMAGE_PATH,RUN)
for index,path in enumerate(paths):
    with Image.open(path) as opened: rgb = np.asarray(ImageOps.exif_transpose(opened).convert('RGB'))
    plt.figure(figsize=(9,6)); plt.imshow(rgb); plt.title(f'{index}: {path.name} — {rgb.shape[1]} x {rgb.shape[0]}'); plt.axis('on'); plt.show()


In [ ]:
# Public checkpoints, no Hugging Face access approval/token needed for these SAM 2.1 weights.
from huggingface_hub import hf_hub_download
from sam2.build_sam import build_sam2, HF_MODEL_ID_TO_FILENAMES
from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator
if MODEL_ID not in HF_MODEL_ID_TO_FILENAMES: raise ValueError('Select an official SAM 2/2.1 model ID')
CACHE = Path(ROOT) / 'pretrained_cache/sam2'
config_name,checkpoint_name = HF_MODEL_ID_TO_FILENAMES[MODEL_ID]
checkpoint = Path(hf_hub_download(MODEL_ID,checkpoint_name,revision=MODEL_REVISION,cache_dir=str(CACHE)))
digest = hashlib.sha256()
with checkpoint.open('rb') as handle:
    for chunk in iter(lambda:handle.read(1024*1024),b''): digest.update(chunk)
model = build_sam2(config_name,str(checkpoint),device=DEVICE,apply_postprocessing=False)
model.eval()
generator = SAM2AutomaticMaskGenerator(model,points_per_side=POINTS_PER_SIDE,
    points_per_batch=POINTS_PER_BATCH,pred_iou_thresh=PRED_IOU_THRESH,
    stability_score_thresh=STABILITY_SCORE_THRESH,crop_n_layers=CROP_N_LAYERS,
    crop_n_points_downscale_factor=2,min_mask_region_area=0,use_m2m=USE_M2M,output_mode='binary_mask')
settings = {'model_id':MODEL_ID,'requested_revision':MODEL_REVISION,'resolved_revision':checkpoint.parent.name,
    'checkpoint':str(checkpoint),'checkpoint_sha256':digest.hexdigest(),'sam2_commit':SAM2_COMMIT,
    'device':DEVICE,'seed':SEED,'points_per_side':POINTS_PER_SIDE,'points_per_batch':POINTS_PER_BATCH,
    'crop_n_layers':CROP_N_LAYERS,'crop_n_points_downscale_factor':2,'pred_iou_thresh':PRED_IOU_THRESH,
    'stability_score_thresh':STABILITY_SCORE_THRESH,'use_m2m':USE_M2M,
    'minimum_area':MIN_MASK_AREA,'maximum_fraction':MAX_MASK_FRACTION,'apply_postprocessing':False}
print(json.dumps(settings,indent=2))


In [ ]:
# Each rerun is a new attempt. Save each completed image immediately.
OUT = RUN / ('sam2_results_' + uuid.uuid4().hex)
OUT.mkdir()
(OUT / 'settings.json').write_text(json.dumps(settings,indent=2))
summaries = []
for index,path in enumerate(paths):
    with Image.open(path) as opened: rgb = np.asarray(ImageOps.exif_transpose(opened).convert('RGB'))
    if GPU: torch.cuda.synchronize()
    start = time.perf_counter()
    with torch.inference_mode(): annotations = generator.generate(rgb)
    if GPU: torch.cuda.synchronize()
    elapsed = time.perf_counter()-start
    folder = OUT / f'image_{index:02d}'
    result,overlay = save_candidates(rgb,annotations,folder,MIN_MASK_AREA,MAX_MASK_FRACTION,SEED)
    Image.fromarray(rgb).save(folder / 'input.png')
    summaries.append({'image':str(path),'output':folder.name,'raw_masks':result['raw_masks'],
                      'kept_candidates':result['kept_candidates'],'inference_seconds':elapsed})
    report = {'model':settings,'images':summaries,'accuracy':'not_measured',
              'latency_note':'One automatic grid/crop pass per image; includes cold-start effects; not webcam FPS',
              'classes':'unknown','quality':'unknown','status':'in_progress'}
    (OUT / 'report.json').write_text(json.dumps(report,indent=2))
    fig,axes = plt.subplots(1,2,figsize=(16,6))
    axes[0].imshow(rgb); axes[0].set_title(path.name)
    axes[1].imshow(overlay); axes[1].set_title(f"{result['kept_candidates']} candidate regions — {elapsed:.1f}s")
    for axis in axes: axis.axis('off')
    plt.show()
    kept = [r for r in result['candidates'] if r['kept']][:MAX_MASK_PREVIEWS]
    if kept:
        fig,axes = plt.subplots((len(kept)+3)//4,4,figsize=(16,4*((len(kept)+3)//4)),squeeze=False)
        for axis in axes.flat: axis.axis('off')
        for axis,row in zip(axes.flat,kept):
            mask = np.asarray(Image.open(folder / row['mask_file'])) > 0
            axis.imshow(np.where(mask[...,None],rgb,255)); axis.set_title(f"ID {row['id']} | predicted IoU {row['model_predicted_iou']:.2f}")
        plt.show()
    del annotations
report['status'] = 'complete'
(OUT / 'report.json').write_text(json.dumps(report,indent=2))
print('Results:',OUT,'| No classification, training, or ground-truth accuracy measurement.')


## Optional: segment a specific part using a box
If automatic masks are poor, set BOX_IMAGE_INDEX and BOX_XYXY in the initial
configuration cell (or directly in a new cell), then run the next cell.
Coordinates are [left, top, right, bottom] in the ORIGINAL photo pixels.
Enclose one visible part, not its whole shadow. Empty BOX_XYXY skips this step.
This is prompted segmentation, not autonomous object detection.


In [ ]:
if BOX_XYXY:
    from sam2.sam2_image_predictor import SAM2ImagePredictor
    if not 0 <= BOX_IMAGE_INDEX < len(paths): raise ValueError('BOX_IMAGE_INDEX is outside the image list')
    with Image.open(paths[BOX_IMAGE_INDEX]) as opened: rgb = np.asarray(ImageOps.exif_transpose(opened).convert('RGB'))
    box = np.asarray(BOX_XYXY,dtype=np.float32)
    if box.shape != (4,) or not np.isfinite(box).all() or not (0 <= box[0] < box[2] <= rgb.shape[1] and 0 <= box[1] < box[3] <= rgb.shape[0]):
        raise ValueError('Use a valid finite [x1,y1,x2,y2] box inside the original image')
    predictor = SAM2ImagePredictor(model)
    with torch.inference_mode():
        predictor.set_image(rgb)
        masks,scores,_ = predictor.predict(box=box,multimask_output=False)
    annotations = [{'segmentation':m,'predicted_iou':float(s)} for m,s in zip(masks,scores)]
    folder = OUT / ('box_prompt_' + uuid.uuid4().hex)
    result,overlay = save_candidates(rgb,annotations,folder,1,1.0,SEED)
    (folder / 'prompt.json').write_text(json.dumps({'image':str(paths[BOX_IMAGE_INDEX]),'box_xyxy':box.tolist()}))
    plt.figure(figsize=(10,7)); plt.imshow(overlay); plt.axis('off'); plt.show()
    print('Prompted masks saved:',folder)
else:
    print('Skipped: set BOX_XYXY to segment a specific part.')


In [ ]:
# Exports preserve binary masks (including holes and overlaps), not lossy polygons.
shutil.copy2(RUN / 'environment.txt',OUT / 'requirements-colab.txt')
shutil.copytree(RUNTIME / 'roboqc',OUT / 'runtime/roboqc')
(OUT / 'README.txt').write_text('SAM 2.1 mask candidates only, not a RoboQC industrial vision bundle. No class/defect labels. Raw masks: raw_masks.npz; filtered masks: PNG; scores/geometry: candidates.json. Checkpoint stays in the Drive cache recorded in settings.json. Images are user-provided; verify redistribution rights.')
archive = shutil.make_archive(str(OUT),'zip',OUT)
print('Download results ZIP from the Files panel or Drive:',archive)
print('This export does not include the large pretrained weights; their cached path/hash are in settings.json.')
